# Упражнение: Безопасность агента: политика вызовов инструментов и «смертельная триада»

## Задание

В `input.csv` — журнал вызовов инструментов агента по 8 сессиям: `call_id, session_id, tool, target, amount`
(`target` — путь, домен, адрес или таблица; `amount` заполнен только у `transfer_money`).
Читайте файл с `keep_default_na=False`, чтобы пустые поля остались пустыми строками.

**Шаг 1. Политика.** Для каждого вызова определите вердикт `allow` (разрешить), `ask` (спросить человека)
или `deny` (запретить) по правилам; применяется **первое** подходящее правило:

1. Неизвестный инструмент (не из списка ниже) — `deny`.
2. Если `target` содержит `..` (выход из каталога) — `deny`.
3. `run_shell`, `delete_db` — `deny`.
4. `read_file`: путь начинается с `/etc/`, содержит `.ssh` или заканчивается на `.env` — `deny`;
   начинается с `/work/` — `allow`; иначе — `ask`.
5. `write_file`: начинается с `/work/tmp/` — `allow`; с `/work/` — `ask`; иначе — `deny`.
6. `query_db`: таблицы `users_pii` и `payments` — `ask`, остальные — `allow`.
7. `read_email` — `allow`.
8. `http_get`: домены `docs.python.org` и `api.company.com` — `allow`, остальные — `ask`.
9. `http_post`: домен `api.company.com` — `ask`, остальные — `deny`.
10. `send_email`: адрес оканчивается на `@company.com` — `ask`, иначе — `deny`.
11. `transfer_money`: сумма ≤ 100 — `ask`, иначе — `deny`.

Список известных инструментов: `read_file, write_file, query_db, read_email, http_get, http_post, send_email,
run_shell, delete_db, transfer_money`.

**Шаг 2. Триада.** Вызов с вердиктом `allow` или `ask` считается доступной возможностью (запрещённый — нет).
Возможности сессии по инструментам:

- `private_data` — `read_file`, `query_db`;
- `untrusted_content` — `http_get`, `read_email`;
- `external_channel` — `http_post`, `send_email`.

Сессия образует «смертельную триаду» (`trifecta = 1`), если у неё есть все три возможности.

## Подсказки

- Правила — это цепочка `if … return` в порядке номеров: сначала проверки 1–3 (они общие), затем разбор по инструменту.
  Если `target` пустой, `..` в нём не встречается.
- Для домена и адреса достаточно сравнения строк: `target in {…}` и `target.endswith('@company.com')`.
  Сумму переводите в число только для `transfer_money`.
- Постройте таблицу вердиктов, затем `groupby('session_id')`: число вердиктов каждого вида и множество возможностей
  из строк с вердиктом не `deny`.

## Ожидаемый формат выходного файла

CSV с колонками `session_id, allow, ask, deny, private_data, untrusted_content, external_channel, trifecta` —
8 строк (сессии 1…8 по порядку). `allow, ask, deny` — числа вызовов с таким вердиктом, остальные колонки — 0 или 1.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/agent_safety
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.